# Dataset Construction
This notebook integrates sequences and generates the positive/negative datasets.

**Option A: Generate Manually**
Run all cells in this notebook to create `negative_ProjectFinal_filtered_score0.1.csv` which is the name of the negative dataset used in this Project.

**Option B: Quick Start**
If you want to skip the generation process, download the final processed file here:
[Download Pre-processed Negative Dataset](https://drive.google.com/file/d/1Aaw5wcOaLjBBvab3sXhod25xtk9jmJUp/view?usp=drive_link)
*Place the downloaded file in this `/Data Files` directory.*

In [ ]:
# This seperates the protein Id and the sequecene from the fasta file and writes it to a text file with Id and sequence seperated by tab.

from Bio import SeqIO
PD_Seq={};s=''
fasta_sequences = SeqIO.parse(open("neg_data.fasta"),'fasta')
#with open("Parse_Fasta.txt") as out_file:
for fasta in fasta_sequences:
    name, sequence = fasta.id, str(fasta.seq)
    #new_sequence = some_function(sequence)
    #write_fasta("Parse_Fasta.txt")
    pid=name.split("\n")[0].split("|")[1]
    PD_Seq[pid]=sequence
print(len(PD_Seq))

for ele in PD_Seq:
    s+="{}\t{}\n".format(ele,PD_Seq[ele])
fp=open("neg_data.txt","w")
fp.write(s);fp.close()
print("===============End==================")


In [ ]:
import pandas as pd

# File paths
csv_path = "positive_interaction_csv.csv" # This is only for positive set. Change the file path while doing for the negative set.
pos_data_path = "Pos_Data.txt"      # Similarly, change it to Neg_Data.txt for negative set.
viral_data_path = "Viral_Data.txt"

# Loading the text files into dictionaries (case-insensitive matching)
def load_sequences(file_path):
    seq_dict = {}
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            if "\t" in line:
                pid, seq = line.split("\t", 1)
            else:
                parts = line.split(maxsplit=1)
                if len(parts) != 2:
                    continue
                pid, seq = parts
            pid, seq = pid.strip().upper(), seq.strip().replace(" ", "")
            if pid not in seq_dict:
                seq_dict[pid] = seq
    return seq_dict

# Loading sequence dictionaries
human_seqs = load_sequences(pos_data_path)
virus_seqs = load_sequences(viral_data_path)

# Loading CSV
df = pd.read_csv(csv_path, dtype=str).fillna("")

# Normalizing ID columns so there is no confilcts for case insensitiveness.
df["Human"] = df["Human"].astype(str).str.strip().str.upper()
df["SARS-CoV2"] = df["SARS-CoV2"].astype(str).str.strip().str.upper()

# Adding sequence columns (leave blank if not found)
df["Human_Sequence"] = df["Human"].map(human_seqs).fillna("")
df["SARS-CoV2_Sequence"] = df["SARS-CoV2"].map(virus_seqs).fillna("")

# Reorder columns
desired_order = ["Human", "Human_Sequence", "SARS-CoV2", "SARS-CoV2_Sequence", "Score"]
for col in df.columns:
    if col not in desired_order:
        desired_order.append(col)

df = df[desired_order]

# Save the final CSV with sequences.
output_path = "positive_ProjectFinal.csv" # Give a different name for the negative file.
df.to_csv(output_path, index=False)

output_path